# Qwen2.5-VL-3B decoder attention on CMU-MOSEI

This Kaggle notebook visualizes how Qwen's **final prompt token** attends to every input token at every language-decoder layer. At each layer, every token receives one value: the arithmetic mean across all decoder attention heads. Only this final-query row is reconstructed; full sequence-by-sequence attention matrices are not retained.

Attach the current `pushkarsingh2005/benchmark-code` and `pushkarsingh2005/cmu-mosei-raw-videos-and-csvs` datasets, enable a GPU and Internet, then run from top to bottom. Attention is an internal routing signal, not a causal explanation.

## 1. Fixed paths and configuration

Set `SAMPLE_IDS` to exact MOSEI IDs for a controlled comparison. When it is empty, the notebook selects high-scoring clips from different emotion dimensions for exploratory visualization; that selection must not be presented as a population-level benchmark.

In [ ]:
from pathlib import Path

CODE = Path('/kaggle/input/datasets/pushkarsingh2005/benchmark-code')
MOSEI_ROOT = Path('/kaggle/input/datasets/pushkarsingh2005/cmu-mosei-raw-videos-and-csvs')
RAW_DIR = MOSEI_ROOT / 'Raw' / 'Raw'
MODEL_ID = 'Qwen/Qwen2.5-VL-3B-Instruct'
SPLIT = 'test'
NUM_CLIPS = 3
SAMPLE_IDS = []  # Example: ['video_id[clip_id]', 'another_id[clip_id]']
FPS = 6.0
MIN_FRAMES = 4
MAX_FRAMES = 64
FRAME_SIZE = 224
SKIP_EXISTING = True

RUN_ROOT = Path('/kaggle/working/experiments/attention_analysis/qwen2_5_vl_3b_mosei_last_query')
INDEX_DIR = RUN_ROOT / 'indexes'
INDEX_CSV = INDEX_DIR / f'mosei_{SPLIT}_index.csv'
OUTPUT_DIR = RUN_ROOT / SPLIT
PREPARE = CODE / 'scripts/data/prepare_mosei.py'
ANALYZER = CODE / 'scripts/analysis/visualize_mosei_decoder_attention.py'

for path in (CODE, MOSEI_ROOT, RAW_DIR, PREPARE, ANALYZER):
    print('Checking:', path, flush=True)
    assert path.exists(), f'Missing {path}. Attach or refresh the Kaggle inputs.'
assert 'arithmetic mean over all language-decoder attention heads' in ANALYZER.read_text(), (
    'Attached benchmark-code is too old; refresh it from the repository.'
)
assert SPLIT in {'train', 'valid', 'test'} and NUM_CLIPS > 0
assert FPS > 0 and 0 < MIN_FRAMES <= MAX_FRAMES and FRAME_SIZE > 0

ALIGNED_CSV = None  # Set an exact Path here only if your CSV is elsewhere.
if ALIGNED_CSV is None:
    candidates = (
        MOSEI_ROOT / 'mosei_aligned.csv',
        MOSEI_ROOT / 'Raw' / 'mosei_aligned.csv',
        MOSEI_ROOT / 'Raw' / 'Raw' / 'mosei_aligned.csv',
    )
    ALIGNED_CSV = next((path for path in candidates if path.is_file()), None)
    assert ALIGNED_CSV is not None, f'Set ALIGNED_CSV manually. Checked: {[str(x) for x in candidates]}'
ALIGNED_CSV = Path(ALIGNED_CSV)
INDEX_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('CSV:', ALIGNED_CSV)
print('Videos:', RAW_DIR)
print('Run root:', RUN_ROOT)


## 2. Install and verify the tested runtime

The analyzer runs in a fresh Python process after installation, matching the repository's extraction notebooks.

In [ ]:
!pip install -q "transformers==4.57.1" "qwen-vl-utils[decord]>=0.0.14" accelerate decord pandas numpy matplotlib Pillow opencv-python-headless


In [ ]:
from importlib.metadata import version
import gc
import hashlib
import json
import os
import shutil
import subprocess
import sys

import numpy as np
import pandas as pd
import torch
from IPython.display import FileLink, Image, display

assert version('transformers') == '4.57.1', 'Restart the session after installing Transformers 4.57.1.'
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator.'
for index in range(torch.cuda.device_count()):
    gpu = torch.cuda.get_device_properties(index)
    print(f'GPU {index}: {gpu.name}, {gpu.total_memory / 2**30:.1f} GiB')


## 3. Build and validate the MOSEI index

The Kaggle videos are stored under `Raw/Raw/<video_id>/<clip_id>.mp4`. The repository indexer validates sample IDs, released split isolation, timestamps, emotion scores, and clip existence. MOSEI's six emotion columns remain continuous/multilabel annotations; they are used here only to describe or select illustrative clips.

In [ ]:
prepare_command = [
    sys.executable, '-u', str(PREPARE),
    '--aligned-csv', str(ALIGNED_CSV),
    '--raw-dir', str(RAW_DIR),
    '--output-dir', str(INDEX_DIR),
]
print('Running:', ' '.join(prepare_command), flush=True)
subprocess.run(prepare_command, check=True)
index = pd.read_csv(INDEX_CSV, dtype={'sample_id': str})
emotions = ['happy', 'sad', 'anger', 'surprise', 'disgust', 'fear']
assert len(index) > 0 and index.sample_id.is_unique
assert index.split.eq(SPLIT).all() and index.video_exists.eq(True).all()
assert index.video_path.map(lambda value: Path(value).is_file()).all()
assert np.isfinite(index[emotions].to_numpy(dtype=float)).all()
print(f'{SPLIT}: {len(index)} verified clips')
display(index[['sample_id', 'utterance', *emotions]].head())


## 4. Select clips

Explicit IDs are preferred for reproducible comparisons. The fallback deliberately selects distinct, high-scoring emotion examples only to make a small qualitative visualization set; it does not estimate dataset-wide behavior.

In [ ]:
if SAMPLE_IDS:
    selected_ids = list(map(str, SAMPLE_IDS))
    missing = [sample_id for sample_id in selected_ids if sample_id not in set(index.sample_id)]
    assert not missing, f'Unknown sample IDs: {missing}'
else:
    selected_ids = []
    for emotion in emotions:
        ranked = index.sort_values([emotion, 'sample_id'], ascending=[False, True])
        candidate = next((sample_id for sample_id in ranked.sample_id.astype(str)
                          if sample_id not in selected_ids), None)
        if candidate is not None:
            selected_ids.append(candidate)
        if len(selected_ids) == NUM_CLIPS:
            break
assert len(selected_ids) == (len(SAMPLE_IDS) if SAMPLE_IDS else NUM_CLIPS)
selected = index.set_index('sample_id').loc[selected_ids].reset_index()
selected.to_csv(RUN_ROOT / 'selected_clips.csv', index=False)
print('Selected IDs:', selected_ids)
display(selected[['sample_id', 'utterance', *emotions]])


## 5. Capture head-averaged final-query attention

For each decoder layer and head, the analyzer reconstructs `softmax(q_last @ K^T / sqrt(d) + mask)` after multimodal RoPE. It then takes the arithmetic mean across heads for each key token and immediately moves that one-dimensional vector to CPU. The prompt and video settings match the Qwen/MOSEI baseline: transcript included, 6 FPS, 4-64 frames, and 224 x 224 processing.

In [ ]:
environment = os.environ.copy()
environment['PYTHONUNBUFFERED'] = '1'
command = [
    sys.executable, '-u', str(ANALYZER),
    '--index-csv', str(INDEX_CSV),
    '--output-dir', str(OUTPUT_DIR),
    '--model-id', MODEL_ID,
    '--fps', str(FPS),
    '--min-frames', str(MIN_FRAMES),
    '--max-frames', str(MAX_FRAMES),
    '--frame-size', str(FRAME_SIZE),
]
if SKIP_EXISTING:
    command.append('--skip-existing')
for sample_id in selected_ids:
    command.extend(['--sample-id', sample_id])
print('Running:', ' '.join(command), flush=True)
subprocess.run(command, check=True, env=environment)


## 6. Verify completion and raw attention

This section rejects missing clips, stale settings, malformed arrays, non-normalized attention, missing layer overlays, or outputs produced by another analyzer revision.

In [ ]:
def safe_name(value):
    import re
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', value).strip('_') or 'sample'

manifest = json.loads((OUTPUT_DIR / 'manifest.json').read_text())
assert [record['sample_id'] for record in manifest] == selected_ids
analyzer_hash = hashlib.sha256(ANALYZER.read_bytes()).hexdigest()
summary_frames = []
for record in manifest:
    assert record['model_id'] == MODEL_ID and record['analyzer_sha256'] == analyzer_hash
    assert record['head_reduction'] == 'arithmetic mean over all language-decoder attention heads'
    assert (record['fps'], record['min_frames'], record['max_frames'], record['frame_size']) == (FPS, MIN_FRAMES, MAX_FRAMES, FRAME_SIZE)
    assert record['query_position'] == record['sequence_tokens'] - 1
    sample_dir = OUTPUT_DIR / safe_name(record['sample_id'])
    with np.load(sample_dir / 'attention_arrays.npz') as arrays:
        mean_attention = arrays['mean_attention']
        video_attention = arrays['video_attention']
        video_mask = arrays['video_mask'].astype(bool)
    assert mean_attention.shape == (record['attention_layers'], record['sequence_tokens'])
    assert video_attention.shape[0] == record['attention_layers']
    assert video_attention.reshape(record['attention_layers'], -1).shape[1] == record['visual_tokens']
    assert video_mask.sum() == record['visual_tokens']
    assert np.isfinite(mean_attention).all() and np.allclose(mean_attention.sum(axis=1), 1.0, atol=2e-4)
    overlays = sorted((sample_dir / 'layer_overlays').glob('layer_*.png'))
    assert len(overlays) == record['attention_layers']
    summary = pd.read_csv(sample_dir / 'layer_summary.csv')
    assert summary.layer.tolist() == list(range(record['attention_layers']))
    summary.insert(0, 'sample_id', record['sample_id'])
    summary_frames.append(summary)
    print(record['sample_id'], 'query token:', repr(record['query_token']),
          '| layers:', record['attention_layers'], '| visual tokens:', record['visual_tokens'])
combined_summary = pd.concat(summary_frames, ignore_index=True)
combined_summary.to_csv(RUN_ROOT / 'combined_layer_summary.csv', index=False)
display(combined_summary.groupby('sample_id')['video_mass'].agg(['mean', 'min', 'max']))


## 7. Inspect layer heatmaps

`modality_attention_by_layer.png` uses raw attention mass and is valid for cross-layer comparison. Temporal, spatial, and frame-overlay colors are normalized within each layer to reveal where that layer's video attention is concentrated.

In [ ]:
for record in manifest:
    sample_dir = OUTPUT_DIR / safe_name(record['sample_id'])
    print('\n', record['sample_id'], '-', record['utterance'])
    display(Image(filename=str(sample_dir / 'modality_attention_by_layer.png')))
    display(Image(filename=str(sample_dir / 'temporal_attention_by_layer.png')))
    display(Image(filename=str(sample_dir / 'spatial_attention_all_layers.png')))
    print('First and last decoder-layer frame overlays:')
    display(Image(filename=str(sample_dir / 'layer_overlays/layer_00.png')))
    display(Image(filename=str(sample_dir / 'layer_overlays' / f"layer_{record['attention_layers'] - 1:02d}.png")))


## 8. Save a download bundle

In [ ]:
run_config = {
    'model_id': MODEL_ID, 'split': SPLIT, 'sample_ids': selected_ids,
    'fps': FPS, 'min_frames': MIN_FRAMES, 'max_frames': MAX_FRAMES,
    'frame_size': FRAME_SIZE, 'query': 'final chat-template prompt token',
    'head_reduction': 'arithmetic mean over all language-decoder attention heads',
    'analyzer_sha256': hashlib.sha256(ANALYZER.read_bytes()).hexdigest(),
    'torch_version': torch.__version__, 'transformers_version': version('transformers'),
}
(RUN_ROOT / 'run_config.json').write_text(json.dumps(run_config, indent=2) + '\n')
archive = shutil.make_archive(str(RUN_ROOT), 'zip', root_dir=RUN_ROOT.parent, base_dir=RUN_ROOT.name)
print('Download bundle:', archive)
display(FileLink(os.path.relpath(archive, Path.cwd())))
gc.collect()
torch.cuda.empty_cache()
